## LG Aimers 9기 — LightGBM + CatBoost OOF 앙상블 (Colab)

LightGBM+MLP 챔피언에서 확인된 시계열 검증 원칙과 `game_type` 분포 변화는 참고하되, 이 노트북은 **LightGBM+CatBoost**라는 별도 모델 계열을 검증한다. 두 모델을 동일한 forward 폴드에서 학습하고 확률 평균·logit 평균을 비교한다.

주 지표는 2025년에 가장 가까운 2023·2024 검증 행의 pooled raw Brier Score다. 2022는 장기 안정성 점검으로 남기고, `R`·`F`, 최신 2024 폴드와 취약 구간을 함께 확인한다. validation 정답 평균을 이용한 oracle shift와 테스트 전체 분포 보정은 사용하지 않는다.

### 0. Drive 마운트와 프로젝트 확인

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import zipfile

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass

candidates = [
    Path.cwd(), Path.cwd().parent,
    Path("/content/Aimers_9th"),
    Path("/content/drive/MyDrive/LG_aimers_9th"),
    Path("/content/drive/MyDrive/Aimers_9th"),
]
PROJECT_ROOT = next(
    (p.resolve() for p in candidates
     if (p / "data/train.csv").exists()
     and (p / "experiments/03_lgbm_catboost_ensemble.py").exists()
     and (p / "src/features.py").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "프로젝트를 찾지 못했습니다. 새 experiments 파일과 최신 src/features.py를 Drive에 올렸는지 확인하세요."
    )
print("PROJECT_ROOT =", PROJECT_ROOT)

### 1. 실행 환경 확인

Colab 기본 NumPy·pandas·scikit-learn은 교체하지 않고 모델 패키지만 설치한다. A100은 CatBoost 학습에 사용하며 LightGBM은 현재 CPU 설정이다.

In [ ]:
install = subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "--disable-pip-version-check", "-q",
     "lightgbm==4.6.0", "catboost==1.2.8"],
    capture_output=True, text=True,
)
if install.returncode != 0:
    print(install.stdout)
    print(install.stderr)
    raise RuntimeError("LightGBM/CatBoost 설치에 실패했습니다.")

import catboost
import lightgbm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

gpu_binary = shutil.which("nvidia-smi")
gpu_available = False
if gpu_binary:
    gpu_check = subprocess.run([gpu_binary, "-L"], capture_output=True, text=True)
    gpu_available = gpu_check.returncode == 0
    if gpu_available:
        print(gpu_check.stdout.strip())
print(
    f"Python={sys.version.split()[0]}",
    f"LightGBM={lightgbm.__version__}",
    f"CatBoost={catboost.__version__}",
    f"GPU={gpu_available}",
    f"CPU threads={os.cpu_count()}",
)

### 2. 실험 설정

2022·2023·2024를 순서대로 검증한다. 각 폴드는 이전 시즌만 학습하고 최근성 half-life 0.5를 적용한다. `game_type`과 `season`은 모델 입력에서 제거하지만, 모든 경기 유형의 과거 행은 학습에 사용한다. 선수 ID는 LightGBM에서는 fold-train 코드로, CatBoost에서는 native categorical로 처리한다.

In [ ]:
VALID_SEASONS = [2022, 2023, 2024]
THREADS = max(1, min(8, os.cpu_count() or 2))
CATBOOST_TASK_TYPE = "GPU" if gpu_available else "CPU"
LGBM_ESTIMATORS = 1500
CATBOOST_ITERATIONS = 1200
FORCE_RERUN = False

if CATBOOST_TASK_TYPE == "CPU":
    print("경고: GPU가 없어 CatBoost CPU 학습을 사용합니다. 실행 시간이 길어질 수 있습니다.")
print({
    "valid_seasons": VALID_SEASONS,
    "threads": THREADS,
    "catboost_task_type": CATBOOST_TASK_TYPE,
    "force_rerun": FORCE_RERUN,
})

### 3. LightGBM·CatBoost forward CV 실행

모델별·폴드별 예측을 Drive에 즉시 저장한다. 중간에 런타임이 끊겨도 `FORCE_RERUN=False`이면 완성된 모델 예측은 검증 후 재사용한다. CatBoost GPU는 `Logloss`로 조기 종료하고, 모델·앙상블 선택은 저장된 OOF의 raw Brier로 수행한다. GPU 연산 특성상 완전한 비트 단위 재현은 보장되지 않는다.

In [ ]:
OUTPUT_ROOT = PROJECT_ROOT / "outputs/lgbm_catboost_ensemble_v1"
cmd = [
    sys.executable, str(PROJECT_ROOT / "experiments/03_lgbm_catboost_ensemble.py"),
    "--threads", str(THREADS),
    "--valid-seasons", *map(str, VALID_SEASONS),
    "--lgbm-estimators", str(LGBM_ESTIMATORS),
    "--catboost-iterations", str(CATBOOST_ITERATIONS),
    "--catboost-task-type", CATBOOST_TASK_TYPE,
    "--output-dir", str(OUTPUT_ROOT),
]
if FORCE_RERUN:
    cmd.append("--force-rerun")
print("실행:", " ".join(cmd))
subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

### 4. 결과 파일 확인

In [ ]:
latest_path = OUTPUT_ROOT / "latest_run.json"
if not latest_path.exists():
    raise FileNotFoundError("latest_run.json이 없습니다. 3번 실행 셀을 먼저 완료하세요.")
with latest_path.open(encoding="utf-8") as f:
    latest = json.load(f)
RUN_DIR = Path(latest["run_dir"])
required_results = [
    "summary.json", "model_summary.csv", "fold_metrics.csv",
    "blend_grid.csv", "cross_fitted_blend_weights.csv",
    "prediction_correlations.csv", "group_metrics.csv",
    "training_metadata.csv",
]
missing = [name for name in required_results if not (RUN_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f"결과 파일이 불완전합니다: {missing}")
print("RUN_DIR =", RUN_DIR)
display(pd.Series(latest["summary"], name="value").to_frame())

### 5. 단독 모델과 앙상블의 2023·2024 pooled 성능

`selected_recent_oof`는 같은 2023·2024 OOF에서 혼합비를 선택한 낙관적 진단값이다. 실제 채택 판단에서는 한 시즌을 제외하고 혼합비를 선택한 `cross_fitted_blend`를 우선 확인한다. Brier는 낮을수록, BSS는 높을수록 좋다.

In [ ]:
model_summary = pd.read_csv(RUN_DIR / "model_summary.csv")
model_summary[[
    "model", "n", "brier", "bss_raw", "roc_auc",
    "target_rate", "prediction_mean", "mean_gap",
]].sort_values("brier")

### 6. 폴드별 안정성과 `R`·`F` 안전장치

In [ ]:
fold_metrics = pd.read_csv(RUN_DIR / "fold_metrics.csv")
fold_all = fold_metrics[
    fold_metrics["partition"].eq("fold")
    & fold_metrics["scope"].eq("all")
]
display(fold_all.pivot_table(
    index="model", columns="valid_season",
    values=["brier", "bss_raw", "mean_gap"], aggfunc="first",
))

guardrail = fold_metrics[
    fold_metrics["partition"].eq("fold")
    & fold_metrics["scope"].isin(["R", "F"])
]
guardrail.pivot_table(
    index=["model", "scope"], columns="valid_season",
    values=["brier", "bss_raw", "mean_gap"], aggfunc="first",
)

### 7. 앙상블 방식과 혼합비 탐색

가로축은 LightGBM 비중이다. 0은 CatBoost 100%, 1은 LightGBM 100%다. 단독 모델 사이에서 곡선이 더 낮아지는 구간이 있어야 앙상블 가치가 있다.

In [ ]:
blend_grid = pd.read_csv(RUN_DIR / "blend_grid.csv")
display(blend_grid.head(12)[[
    "method", "lgbm_weight", "catboost_weight",
    "selection_brier", "selection_bss_raw", "latest_brier",
    "worst_selection_fold_brier", "selection_mean_gap",
]])

fig, ax = plt.subplots(figsize=(8, 4))
for method, part in blend_grid.sort_values("lgbm_weight").groupby("method"):
    ax.plot(part["lgbm_weight"], part["selection_brier"], marker="o", ms=3, label=method)
ax.set_xlabel("LightGBM weight")
ax.set_ylabel("2023-2024 pooled OOF Brier (lower is better)")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

### 8. 교차 적합 혼합비와 예측 상관도

각 holdout 시즌의 혼합비는 나머지 시즌 OOF에서만 선택한다. 상관도가 1에 매우 가까우면 두 모델이 비슷한 실수를 하므로 앙상블 이득이 제한적일 수 있다.

In [ ]:
meta_weights = pd.read_csv(RUN_DIR / "cross_fitted_blend_weights.csv")
correlations = pd.read_csv(RUN_DIR / "prediction_correlations.csv")
display(meta_weights)
correlations

### 9. 취약 구간 비교

표본 1,000행 이상에서 BSS가 낮은 구간을 확인한다. 하위 집단 BSS는 공식 점수처럼 합산하지 않고 실패 원인을 찾는 용도로만 사용한다.

In [ ]:
groups = pd.read_csv(RUN_DIR / "group_metrics.csv")
material = groups[
    groups["model"].isin(["lightgbm", "catboost", "cross_fitted_blend"])
    & ~groups["group"].eq("overall")
    & groups["n"].ge(1000)
].copy()
material["mean_gap_pct_point"] = 100 * material["mean_gap"]
material.sort_values(["bss_raw", "n"], ascending=[True, False])[[
    "model", "valid_season", "group", "value", "n",
    "brier", "bss_raw", "roc_auc", "mean_gap_pct_point",
]].head(80)

### 10. 자동 판정 요약

교차 적합 앙상블이 가장 좋은 단독 모델보다 낮은 Brier를 보이는지와 폴드별 개선 방향을 요약한다. 한 번의 작은 차이만으로 최종 모델을 확정하지 않는다.

In [ ]:
pooled = model_summary.set_index("model")
best_base_name = pooled.loc[["lightgbm", "catboost"], "brier"].idxmin()
best_base_brier = pooled.loc[best_base_name, "brier"]
cross_brier = pooled.loc["cross_fitted_blend", "brier"]
pooled_gain = best_base_brier - cross_brier

fold_pivot = fold_all.pivot(index="valid_season", columns="model", values="brier")
fold_pivot["best_base"] = fold_pivot[["lightgbm", "catboost"]].min(axis=1)
fold_pivot["cross_fitted_gain"] = (
    fold_pivot["best_base"] - fold_pivot["cross_fitted_blend"]
)
print(f"최고 단독 모델: {best_base_name} | 2023-2024 pooled Brier={best_base_brier:.9f}")
print(f"교차 적합 앙상블 2023-2024 pooled Brier={cross_brier:.9f}")
print(f"앙상블 개선량(양수=개선): {pooled_gain:+.9f}")
display(fold_pivot[["lightgbm", "catboost", "cross_fitted_blend", "cross_fitted_gain"]])
if pooled_gain > 0 and (fold_pivot["cross_fitted_gain"] >= 0).sum() >= 2:
    print("판정: 앙상블 후보 유지. 다음 단계에서 모델별 소규모 튜닝과 calibration을 검토합니다.")
else:
    print("판정: 현재 혼합은 안정적 개선 근거가 부족합니다. 단독 모델/피처 구성을 먼저 개선합니다.")

### 11. 검토용 결과 묶음 생성

이 단계는 검증 실험이므로 제출용 모델 ZIP을 만들지 않는다. 아래 ZIP에는 결과 검토에 필요한 작은 JSON·CSV만 포함한다. 실행 후 이 파일과 실행 완료된 노트북을 전달하면 다음 실험을 결정할 수 있다.

In [ ]:
review_names = [
    "summary.json", "configuration.json", "model_summary.csv",
    "fold_metrics.csv", "blend_grid.csv",
    "cross_fitted_blend_weights.csv", "prediction_correlations.csv",
    "group_metrics.csv", "training_metadata.csv",
]
REVIEW_ZIP = OUTPUT_ROOT / f"review_bundle_{latest['signature']}.zip"
with zipfile.ZipFile(REVIEW_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for name in review_names:
        zf.write(RUN_DIR / name, arcname=name)
print("검토용 ZIP =", REVIEW_ZIP)